# Kuramoto model: spontaneous synchronisation

*others · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/kuramoto_sync)

Fireflies flashing together, metronomes on a shared board, pacemaker cells: oscillators with different natural frequencies $\omega_i$, each pulled towards the others, suddenly lock when the coupling $K$ passes a threshold. The complex mean field $r e^{i\psi} = \frac1N \sum_j e^{i\theta_j}$ turns the all-to-all sum into $O(N)$ work, a whole batch of couplings runs at once, and the order parameter $r$ jumps up at $K_c = 2/(\pi g(0))$, following the exact Ott–Antonsen curve $r = \sqrt{1 - K_c/K}$ for Lorentzian frequencies.

**Vector form**

$$\dot\theta_i = \omega_i + \frac KN \sum_{j=1}^N \sin(\theta_j - \theta_i) = \omega_i + K\,r\,\sin(\psi - \theta_i),\qquad r e^{i\psi} = \frac1N\sum_j e^{i\theta_j}$$

**Index (tensor) form**

$$z_b = \tfrac1N\, e^{i\theta_{bj}}\,\mathbb 1_j,\qquad \dot\theta_{bi} = \omega_i + K_b\,\operatorname{Im}\!\big(z_b\,e^{-i\theta_{bi}}\big),\qquad K_c = \frac{2}{\pi g(0)},\qquad r_\infty = \sqrt{1 - K_c/K}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\omega_i$ | `frequencies('lorentzian', N, gamma, device, seed)` | (N,) | natural frequencies, one random draw in each of $N$ equal-probability bins of the Lorentzian (or Gaussian) density |
| $K_b$ | `torch.linspace(0, k_max, B)[:, None]` | (B, 1) | one coupling per batch member: every population evolves in parallel |
| $z_b = r_b e^{i\psi_b}$ | `torch.exp(1j * theta).mean(-1, keepdim=True)` | (B, 1) | complex mean field, repeated $j$ ⇒ averaged; $r = 0$ incoherent, $r = 1$ perfect synchrony |
| $\dot\theta_{bi}$ | `omega + k * (z * torch.exp(-1j * theta)).imag` | (B, N) | each oscillator only feels the mean field: $O(N)$ instead of $O(N^2)$ |
| $\sin(\theta_j - \theta_i)$ | `einsum('bij->bi', sin(theta[:, None] - theta[:, :, None]))` | (B, N, N) | the slow pairwise sum, kept as a reference; the tests check it equals the mean-field form |
| $r_\infty$ | `ott_antonsen_r(K, gamma)` | (B,) | exact $N \to \infty$ order parameter for Lorentzian frequencies, $\sqrt{1 - 2\gamma/K}$ above $K_c = 2\gamma$ |
| $G(x_i - x_j)$ | `ring_kernel(x, A)` | (N, N) | nonlocal coupling $(1 + A\cos x)/2\pi$ on a ring, for the chimera state |

Kuramoto (1975) solved the model self-consistently and found $K_c = 2/(\pi g(0))$ for a symmetric unimodal frequency density $g$; Strogatz (2000) tells the story. Ott and Antonsen (2008) proved that for Lorentzian $g(\omega) = (\gamma/\pi)/(\omega^2 + \gamma^2)$ the infinite-$N$ dynamics collapses onto one equation, $\dot r = -\gamma r + \tfrac K2 r(1 - r^2)$, whose stable state is $r = \sqrt{1 - 2\gamma/K}$: the simulation matches it to about $0.01$ with $N = 1200$. Below $K_c$, $r$ is only finite-size noise of order $1/\sqrt N$. Gaussian frequencies have the smaller threshold $K_c = 2\sigma\sqrt{2\pi}/\pi \approx 1.6\,\sigma$ (green). The slow ramp (red) follows the curve as $K$ grows; the circle shows the oscillators in the frame of the mean field, coloured by $\omega_i$: the ones with small $|\omega_i|$ lock first, the fast ones keep drifting. The third panel is a short extra: identical oscillators on a ring with nonlocal coupling and a phase lag $\alpha = \pi/2 - 0.18$ split into a locked and an incoherent domain, the chimera state found by Kuramoto and Battogtokh (2002) and named by Abrams and Strogatz (2004). The mean phase drifts exactly at the mean frequency because $\sin(\theta_j - \theta_i)$ is antisymmetric.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/kuramoto_sync/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import critical_coupling, local_order, ott_antonsen_r


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    n = 300 if fast else params["n_oscillators"]
    width, k_max = params["width"], params["k_max"]
    n_k = 8 if fast else params["n_couplings"]
    t_relax = 10.0 if fast else params["t_relax"]
    dt = 0.1

    k = tensors.couplings(k_max, n_k, dev)  # (B, 1) one population per coupling
    sweeps = {}
    for dist in ("lorentzian", "gaussian"):
        omega = tensors.frequencies(dist, n, width, dev, params["seed"]).float()  # (N,)
        _, r = solve.sweep(omega, k, t_relax, dt, params["seed"] + 1)  # r (B, T + 1)
        r_late = r[:, r.shape[1] // 2 :].mean(1)  # time average after the transient
        k_c = critical_coupling(dist, width)
        sweeps[dist] = (r_late.cpu(), k_c)
        below = r_late[k[:, 0] < 0.8 * k_c]
        print(f"{dist}: N = {n}, width {width}, K_c = 2/(π g(0)) = {k_c:.3f}")
        print(f"  below 0.8 K_c: mean r = {below.mean():.3f}  (finite-size level 1/√N = {1 / math.sqrt(n):.3f})")
        if dist == "lorentzian":
            exact = ott_antonsen_r(k[:, 0].double(), width).float()
            above = k[:, 0] > 1.3 * k_c
            err = (r_late[above] - exact[above]).abs().max()
            print(f"  above 1.3 K_c: max |r − √(1 − K_c/K)| = {err:.3f}  (Ott–Antonsen, exact for N → ∞)")
            for kk, rr, ee in list(zip(k[:, 0].tolist(), r_late.tolist(), exact.tolist()))[:: max(1, n_k // 6)]:
                print(f"    K = {kk:5.2f}: r = {rr:.3f}   Ott–Antonsen {ee:.3f}")

    dist = params["distribution"]
    omega = tensors.frequencies(dist, n, width, dev, params["seed"] + 2).float()
    ramp_time = 40.0 if fast else params["ramp_time"]
    k_t, thetas, r_t = solve.ramp(omega, k_max, ramp_time, dt, 10 if fast else 60, params["seed"] + 3)
    print(f"slow ramp of K from 0 to {k_max} over t = {ramp_time:.0f} ({dist}): final r = {r_t[-1]:.3f}")

    x, theta_ring = solve.chimera(n=128 if fast else 256, t_max=30.0 if fast else 300.0, dt=0.05, seed=params["seed"])
    loc = local_order(theta_ring, 5)
    print(
        f"chimera ring (identical oscillators, A = 0.995, α = π/2 − 0.18): local order ranges from "
        f"{loc.min():.2f} (drifting part) to {loc.max():.2f} (locked part)"
    )
    return plot.animate(
        omega.cpu(),
        k_t,
        thetas.cpu(),
        r_t.cpu(),
        k[:, 0].cpu(),
        sweeps,
        width,
        dist,
        (x, theta_ring),
        out_dir / "kuramoto.gif",
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/kuramoto_sync/equation.py
r"""Coupled oscillators → tensor translation.

Kuramoto model (Kuramoto 1975). N oscillators with phases θ_i and natural frequencies ω_i, all coupled to all:

    \dot\theta_i = \omega_i + \frac KN \sum_j \sin(\theta_j - \theta_i).

The complex mean field (order parameter) r e^{iψ} = (1/N) Σ_j e^{iθ_j} turns the O(N²) double sum into O(N):

    \dot\theta_i = \omega_i + K\,r\,\sin(\psi - \theta_i)
                 = \omega_i + K\,\operatorname{Im}\!\big(r e^{i\psi} e^{-i\theta_i}\big).

Each oscillator only feels the mean field, which in turn is made of the oscillators: r ≈ 0 means incoherence,
r = 1 perfect synchrony. With a symmetric unimodal frequency density g, the incoherent state loses stability at
K_c = 2 / (π g(0)) (Kuramoto). For the Lorentzian g(ω) = (γ/π)/(ω² + γ²), K_c = 2γ, and Ott and Antonsen (2008)
proved that the N → ∞ dynamics reduces exactly to one equation for r, whose stable state is r = √(1 − K_c/K).

A ring with nonlocal coupling G and a phase lag α (Kuramoto and Battogtokh 2002, Abrams and Strogatz 2004),

    \dot\theta_i = \omega - \frac{2\pi}N \sum_j G(x_i - x_j)\,\sin(\theta_i - \theta_j + \alpha),

can split identical oscillators into a locked domain and a drifting one: a chimera state.
"""

import math

import torch


def order_parameter(theta: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""r e^{iψ} = (1/N) Σ_j e^{iθ_j}   (j summed).   (..., N) → r (...), ψ (...)"""
    z = torch.exp(1j * theta).mean(-1)
    return z.abs(), z.angle()


def velocity(theta: torch.Tensor, omega: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    r"""θ̇_i = ω_i + K Im(z e^{−iθ_i}) with z = r e^{iψ}: one batch member per coupling K.
    (B, N), (N,) or (B, N), (B, 1) → (B, N)"""
    z = torch.exp(1j * theta).mean(-1, keepdim=True)  # (B, 1) complex mean field
    return omega + k * (z * torch.exp(-1j * theta)).imag


def pairwise_velocity(theta: torch.Tensor, omega: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    r"""Slow reference: θ̇_i = ω_i + (K/N) Σ_j sin(θ_j − θ_i)   (j summed over an (N, N) matrix).
    (B, N), (N,) or (B, N), (B, 1) → (B, N)"""
    s = torch.sin(theta[:, None, :] - theta[:, :, None])  # (B, N_i, N_j): sin(θ_j − θ_i)
    return omega + k / theta.shape[-1] * torch.einsum("bij->bi", s)


def critical_coupling(distribution: str, width: float) -> float:
    """K_c = 2 / (π g(0)): Lorentzian half-width γ → 2γ; Gaussian standard deviation σ → 2σ√(2π)/π ≈ 1.596 σ."""
    g0 = 1 / (math.pi * width) if distribution == "lorentzian" else 1 / (width * math.sqrt(2 * math.pi))
    return 2 / (math.pi * g0)


def ott_antonsen_r(k: torch.Tensor, gamma: float) -> torch.Tensor:
    """Exact N → ∞ order parameter for Lorentzian frequencies: r = √(1 − 2γ/K) above K_c = 2γ, else 0.   (B,) → (B,)"""
    return (1 - 2 * gamma / k).clamp(min=0).sqrt()


def ring_kernel(x: torch.Tensor, a: float) -> torch.Tensor:
    """G_ij = (1 + A cos(x_i − x_j)) / 2π on the ring x ∈ [−π, π).   (N,) → (N, N)"""
    return (1 + a * torch.cos(x[:, None] - x[None, :])) / (2 * math.pi)


def ring_velocity(theta: torch.Tensor, g: torch.Tensor, alpha: float, omega: float) -> torch.Tensor:
    r"""θ̇_i = ω − Im(e^{i(θ_i + α)} conj(Z_i)), Z_i = (2π/N) Σ_j G_ij e^{iθ_j}   (j summed), because
    sin(θ_i − θ_j + α) = Im(e^{i(θ_i+α)} e^{−iθ_j}).   (N,), (N, N) → (N,)"""
    z = (2 * math.pi / theta.shape[-1]) * torch.einsum("ij,j->i", g.to(torch.complex128), torch.exp(1j * theta))
    return omega - (torch.exp(1j * (theta + alpha)) * z.conj()).imag


def local_order(theta: torch.Tensor, half_width: int) -> torch.Tensor:
    """|mean of e^{iθ_j}| over the 2w + 1 nearest neighbours on the ring: ≈ 1 in a locked domain.   (N,) → (N,)"""
    z = torch.exp(1j * theta)
    window = torch.stack([torch.roll(z, s) for s in range(-half_width, half_width + 1)])  # (2w+1, N)
    return window.mean(0).abs()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/kuramoto_sync/tensors.py
"""Build the concrete tensors: natural frequencies, random initial phases, couplings and the ring."""

import math

import torch


def frequencies(distribution: str, n: int, width: float, device: torch.device, seed: int) -> torch.Tensor:
    """n natural frequencies ω_i with zero mean: Lorentzian of half-width γ or Gaussian of standard deviation σ.
    Stratified sampling: one random draw inside each of n equal-probability bins, u_i = (i + U_i)/n, then the inverse
    CDF. This keeps the finite-N sample close to the true density (fewer accidental clusters).   → (n,) float64"""
    gen = torch.Generator().manual_seed(seed)
    u = (
        torch.arange(n, dtype=torch.float64) + torch.rand(n, generator=gen, dtype=torch.float64).clamp(1e-3, 1 - 1e-3)
    ) / n
    if distribution == "lorentzian":
        omega = width * torch.tan(math.pi * (u - 0.5))  # inverse CDF of the Cauchy law
    else:
        omega = width * math.sqrt(2) * torch.erfinv(2 * u - 1)  # inverse CDF of the normal law
    return omega[torch.randperm(n, generator=gen)].to(device)


def initial_phases(batch: int, n: int, device: torch.device, seed: int) -> torch.Tensor:
    """Independent uniform phases in [0, 2π) for every batch member: (B, N) float64."""
    gen = torch.Generator().manual_seed(seed)
    return (2 * math.pi * torch.rand(batch, n, generator=gen, dtype=torch.float64)).to(device)


def couplings(k_max: float, count: int, device: torch.device) -> torch.Tensor:
    """count coupling strengths from 0 to k_max, one per batch member: (B, 1)."""
    return torch.linspace(0, k_max, count, dtype=torch.float64, device=device)[:, None]


def ring(n: int, device: torch.device) -> torch.Tensor:
    """Positions x_j = −π + 2πj/n on the ring: (n,) float64."""
    return -math.pi + 2 * math.pi * torch.arange(n, dtype=torch.float64, device=device) / n


def chimera_start(x: torch.Tensor, seed: int) -> torch.Tensor:
    """Abrams and Strogatz's initial condition θ(x) = 6 ρ e^{−0.76 x²}, ρ uniform in [−½, ½]: a random bump.   (N,)"""
    gen = torch.Generator().manual_seed(seed)
    rho = torch.rand(x.shape, generator=gen, dtype=torch.float64).to(x.device) - 0.5
    return 6 * rho * torch.exp(-0.76 * x**2)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/kuramoto_sync/solve.py
"""Time stepping (RK4): a batch of couplings at once, a slow ramp of K, and the chimera ring."""

import math

import torch

from problems._common.integrators import rk4_step
from problems._common.progress import steps

from . import tensors
from .equation import order_parameter, ring_kernel, ring_velocity, velocity


def integrate(
    theta0: torch.Tensor, omega: torch.Tensor, k: torch.Tensor, t_max: float, dt: float, save_every: int = 1
) -> tuple[torch.Tensor, torch.Tensor]:
    """RK4 for every coupling K_b at once. Phases are not wrapped (so the mean phase can be followed).
    theta0 (B, N), omega (N,), k (B, 1) → saved phases (S, B, N), order parameter at every step (B, T + 1)."""
    n_steps = round(t_max / dt)
    theta = theta0.to(omega.dtype)
    saved, rs = [theta], [order_parameter(theta)[0]]
    for i in steps(n_steps, "RK4 steps"):
        theta = rk4_step(lambda th, t: velocity(th, omega, k), theta, i * dt, dt)
        rs.append(order_parameter(theta)[0])
        if (i + 1) % save_every == 0:
            saved.append(theta)
    return torch.stack(saved), torch.stack(rs, dim=1)


def sweep(
    omega: torch.Tensor, k: torch.Tensor, t_max: float, dt: float, seed: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """One independent population per coupling K_b, all from random phases.   (N,), (B, 1) → final phases (B, N),
    r(t) (B, T + 1)"""
    theta0 = tensors.initial_phases(len(k), len(omega), omega.device, seed)
    saved, r = integrate(theta0, omega, k.to(omega.dtype), t_max, dt, save_every=round(t_max / dt))
    return saved[-1], r


def ramp(
    omega: torch.Tensor, k_max: float, t_max: float, dt: float, n_frames: int, seed: int
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """One population while K grows slowly from 0 to k_max: K(t) = k_max t / t_max.
    → K at the saved frames (F,), phases (F, N), r at the frames (F,)."""
    n_steps = round(t_max / dt)
    every = max(1, n_steps // n_frames)
    theta = tensors.initial_phases(1, len(omega), omega.device, seed).to(omega.dtype)
    ks, thetas = [0.0], [theta[0]]
    for i in steps(n_steps, "ramp"):
        theta = rk4_step(lambda th, t: velocity(th, omega, torch.tensor(k_max * t / t_max)), theta, i * dt, dt)
        if (i + 1) % every == 0:
            ks.append(k_max * (i + 1) / n_steps)
            thetas.append(theta[0])
    thetas = torch.stack(thetas)
    return torch.tensor(ks), thetas, order_parameter(thetas)[0]


def chimera(
    n: int = 256, t_max: float = 300.0, dt: float = 0.05, seed: int = 0, a: float = 0.995, beta: float = 0.18
) -> tuple[torch.Tensor, torch.Tensor]:
    """Identical oscillators on a ring with kernel (1 + A cos x)/2π and phase lag α = π/2 − β (Abrams and Strogatz
    2004). Returns positions (N,) and the final phases wrapped to (−π, π] (N,), in the frame of the locked domain."""
    x = tensors.ring(n, torch.device("cpu"))
    g = ring_kernel(x, a)
    alpha = math.pi / 2 - beta
    theta = tensors.chimera_start(x, seed)
    for i in steps(round(t_max / dt), "ring"):
        theta = rk4_step(lambda th, t: ring_velocity(th, g, alpha, 0.0), theta, i * dt, dt)
    z = torch.exp(1j * theta)
    local = torch.stack([torch.roll(z, s) for s in range(-5, 6)]).mean(0)  # (N,) neighbourhood mean field
    locked = local[local.abs().argmax()].angle()  # phase of the most coherent neighbourhood
    return x, torch.remainder(theta - locked + math.pi, 2 * math.pi) - math.pi


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/kuramoto_sync/plot.py
"""Dots on the unit circle clustering as K ramps up, beside r(K) and a chimera snapshot on a ring."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import ott_antonsen_r

SHOWN = 400  # oscillators drawn on the circle


def animate(
    omega: torch.Tensor,
    k_t: torch.Tensor,
    thetas: torch.Tensor,
    r_t: torch.Tensor,
    k: torch.Tensor,
    sweeps: dict,
    width: float,
    dist: str,
    ring: tuple[torch.Tensor, torch.Tensor],
    path: Path,
) -> Path:
    """omega (N,); k_t, r_t (F,) and thetas (F, N) along the ramp; k (B,) and sweeps {dist: (r (B,), K_c)}."""
    th = thetas[:, :SHOWN].numpy()
    psi = torch.exp(1j * thetas).mean(-1).angle().numpy()  # (F,) mean-field direction
    w = omega[:SHOWN].clamp(-3 * width, 3 * width).numpy()
    kk, rr = k_t.numpy(), r_t.numpy()

    fig = plt.figure(figsize=(11, 4))
    grid = fig.add_gridspec(1, 3, width_ratios=[1, 1.25, 1.1], wspace=0.32)
    ax_c = fig.add_subplot(grid[0])
    circle = np.linspace(0, 2 * np.pi, 200)
    ax_c.plot(np.cos(circle), np.sin(circle), color="#d1d5db", lw=1)
    dots = ax_c.scatter(np.cos(th[0]), np.sin(th[0]), c=w, cmap="coolwarm", s=9, vmin=-3 * width, vmax=3 * width)
    (arrow,) = ax_c.plot([0, 0], [0, 0], color="#111827", lw=2.5)
    ax_c.set_xlim(-1.15, 1.15)
    ax_c.set_ylim(-1.15, 1.15)
    ax_c.set_aspect("equal")
    ax_c.set_xticks([])
    ax_c.set_yticks([])
    ax_c.grid(False)
    title = ax_c.set_title("", fontsize=10)
    ax_c.text(0, -1.32, "colour: natural frequency ω (mean field → up)", ha="center", fontsize=7, color="#6b7280")

    ax_r = fig.add_subplot(grid[1])
    k_fine = torch.linspace(1e-3, float(k.max()), 300)
    ax_r.plot(k_fine, ott_antonsen_r(k_fine, width), color="#111827", lw=1.5, label="Ott–Antonsen √(1 − K_c/K)")
    r_l, kc_l = sweeps["lorentzian"]
    r_g, kc_g = sweeps["gaussian"]
    ax_r.plot(k, r_l, "o", color="#2563eb", ms=4, label="Lorentzian (batch of K)")
    ax_r.plot(k, r_g, "s", color="#16a34a", ms=3.5, mfc="none", label="Gaussian (batch of K)")
    ax_r.axvline(kc_l, color="#2563eb", ls="--", lw=1)
    ax_r.axvline(kc_g, color="#16a34a", ls=":", lw=1)
    (trace,) = ax_r.plot([], [], color="#dc2626", lw=1.3, label=f"slow ramp ({dist})")
    (dot,) = ax_r.plot([], [], "o", color="#dc2626", ms=6)
    ax_r.set_xlim(0, float(k.max()))
    ax_r.set_ylim(0, 1.02)
    ax_r.set_xlabel("coupling K")
    ax_r.set_ylabel("order parameter r")
    ax_r.set_title(f"phase transition at K_c = 2/(π g(0))  ({kc_l:.2f}, {kc_g:.2f})", fontsize=9)
    ax_r.legend(fontsize=7, frameon=False, loc="lower right")

    x, theta = ring
    ax_x = fig.add_subplot(grid[2])
    ax_x.plot(x.numpy(), theta.numpy(), ".", color="#9333ea", ms=3)
    ax_x.set_xlim(-np.pi, np.pi)
    ax_x.set_ylim(-np.pi, np.pi)
    ax_x.set_xlabel("position on the ring x")
    ax_x.set_ylabel("phase θ")
    ax_x.set_title("chimera: identical oscillators,\none locked domain beside a drifting one", fontsize=9)

    def update(f: int):
        a = th[f] - psi[f] + np.pi / 2  # rotate so that the mean field points up
        dots.set_offsets(np.c_[np.cos(a), np.sin(a)])
        arrow.set_data([0, 0], [0, rr[f]])
        title.set_text(f"K = {kk[f]:.2f},  r = {rr[f]:.2f}")
        trace.set_data(kk[: f + 1], rr[: f + 1])
        dot.set_data([kk[f]], [rr[f]])

    return save_gif(fig, update, frame_indices(len(kk), 60), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_oscillators': 1200,
    'width': 0.5,
    'k_max': 3.0,
    'n_couplings': 24,
    't_relax': 60.0,
    'ramp_time': 200.0,
    'distribution': 'lorentzian',
    'seed': 0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.kuramoto_sync")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.kuramoto_sync.main")
problem_dir = Path("problems/others/kuramoto_sync")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
